In [1]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [9]:
del work_dir, image_root

In [10]:
# Set the working directory
import os
import numpy as np
import pandas as pd

#work_dir = os.path.dirname(os.path.dirname(os.getcwd())) 
work_dir = os.path.dirname(os.getcwd())

print(f"Current working directory: {work_dir}")

Current working directory: c:\Users\dysk-\Desktop\Current task\EEG compe


# Set Image Dir

In [11]:
image_dir = os.path.join(work_dir, "data", "training_images")

print(f"Image directory: {image_dir}")

Image directory: c:\Users\dysk-\Desktop\Current task\EEG compe\data\training_images


# Read image_paths.txt

In [14]:
from pathlib import Path

image_root = work_dir + "/data" + "/training_images"

imagelist_path = work_dir + "/data" + "/train" + "/image_paths.txt"

with open(imagelist_path, "r") as f:
    paths = [p.strip() for p in f.readlines()]

print(len(paths))
print(paths[0])

118800
00001_aardvark/aardvark_01b.jpg


# ViT

In [18]:
import torch
from PIL import Image
from torchvision.models import vit_b_16, ViT_B_16_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

weights = ViT_B_16_Weights.IMAGENET1K_V1
vit = vit_b_16(weights=weights)
vit.heads = torch.nn.Identity()
vit = vit.to(device).eval()

preprocess = weights.transforms()

# 1枚テスト
img_path = image_dir + "/" + paths[0]
print(img_path)
print(Path(img_path).exists())

img = Image.open(img_path).convert("RGB")
x = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    feat = vit(x)

print(feat.shape)

c:\Users\dysk-\Desktop\Current task\EEG compe\data\training_images/00001_aardvark/aardvark_01b.jpg
True
torch.Size([1, 768])


# Extract ViT feature

In [19]:
from tqdm.notebook import tqdm
import numpy as np
from pathlib import Path

# 重複を除いた画像path
unique_paths = sorted(list(set(paths)))

print("unique images:", len(unique_paths))
print(unique_paths[0])

features = []

vit.eval()

with torch.no_grad():
    for rel_path in tqdm(unique_paths):
        img_path = Path(image_dir) / rel_path

        img = Image.open(img_path).convert("RGB")
        x = preprocess(img).unsqueeze(0).to(device)

        feat = vit(x)
        features.append(feat.cpu().numpy()[0])

features = np.stack(features)

print("features shape:", features.shape)

unique images: 5940
00001_aardvark/aardvark_01b.jpg


  0%|          | 0/5940 [00:00<?, ?it/s]

features shape: (5940, 768)


In [20]:
out_dir = Path("features")
out_dir.mkdir(exist_ok=True)

np.save(out_dir / "vit_image_features.npy", features)

with open(out_dir / "vit_image_paths.txt", "w") as f:
    for p in unique_paths:
        f.write(p + "\n")

print("saved")

saved


In [22]:
from pathlib import Path
import numpy as np

feature_path = work_dir + "/data/features/vit_image_features.npy"
path_txt = work_dir + "/data/features/vit_image_paths.txt"
print(feature_path)


features = np.load(feature_path)

with open(path_txt) as f:
    feature_paths = [p.strip() for p in f.readlines()]

print(features.shape)
print(len(feature_paths))
print(feature_paths[0])

c:\Users\dysk-\Desktop\Current task\EEG compe/data/features/vit_image_features.npy
(5940, 768)
5940
00001_aardvark/aardvark_01b.jpg


In [23]:
feature_dict = {
    p: feat
    for p, feat in zip(feature_paths, features)
}

print(len(feature_dict))
print(feature_dict[feature_paths[0]].shape)

5940
(768,)


In [24]:
train_path_file = work_dir + "/data/train/image_paths.txt"

with open(train_path_file) as f:
    train_paths = [p.strip() for p in f.readlines()]

print(len(train_paths))
print(train_paths[0])

118800
00001_aardvark/aardvark_01b.jpg


In [25]:
train_image_features = np.stack([
    feature_dict[p]
    for p in train_paths
])

print(train_image_features.shape)

(118800, 768)
